# Notebook 05 — Ablation Study + SHAP

## Diseño del experimento de ablación

Un **ablation study** mide el impacto de cada componente del sistema eliminándolo o añadiéndolo uno a la vez. En nuestro caso queremos responder:

1. ¿Mejora el sentiment la predicción de XGBoost? (Exp 2 vs Exp 3)
2. ¿Puede el MLP igualar o superar al XGBoost? (Exp 3 vs Exp 4)
3. ¿Qué features son más importantes para el mejor modelo?

| Exp | Modelo              | Features                              | Notebook |
|-----|---------------------|---------------------------------------|----------|
| 1   | Logistic Regression | Técnicas + Macro (11)                 | 03       |
| 2   | XGBoost             | Técnicas + Macro (11)                 | 03       |
| 3   | XGBoost             | Técnicas + Macro + Sentiment (14)     | **05**   |
| 4   | MLP                 | Técnicas + Macro + Sentiment (14)     | **05**   |

**Input**: `data/processed/sp500_features.csv` + sentiment agregado del cache de FinBERT.

**Output**: `data/processed/feature_matrix.csv` con las 14 features + target, listo para los experimentos 3 y 4.

In [ ]:
import sys
sys.path.insert(0, "..")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src import data_loader, features, sentiment, models, utils

utils.set_plot_style()

RAW_DIR       = "../data/raw"
PROCESSED_DIR = "../data/processed"
MODELS_DIR    = "../models"
CACHE_PATH    = f"{PROCESSED_DIR}/sentiment_cache.csv"

FEATURE_COLS_BASE = [
    "RSI_14", "MACD", "BB_position", "return_1d", "return_5d", "volume_change",
    "vix", "t10y2y", "fedfunds", "cpi", "unrate"
]
FEATURE_COLS_SENT = FEATURE_COLS_BASE + ["sentiment_mean", "sentiment_std", "news_count"]

## Construcción de la feature matrix con sentiment

Cargamos el dataset base (`sp500_features.csv`) y lo enriquecemos con el sentiment diario agregado y shifteado 1 día. El resultado es la `feature_matrix.csv` con las 14 features finales + la variable objetivo.

Los días sin cobertura de noticias (feriados, fines de semana donde se publicaron noticias pero el mercado no abrió, o días muy tempranos con poca cobertura) reciben `sentiment_mean=0`, `sentiment_std=0` y `news_count=0` — lo que representa neutralidad de información.

In [ ]:
# Cargar datos base
sp500_features_df = pd.read_csv(f"{PROCESSED_DIR}/sp500_features.csv",
                                index_col=0, parse_dates=True)

# Cargar y agregar sentiment
news = data_loader.load_news(RAW_DIR)
raw_sent = sentiment.load_or_run_finbert(news, CACHE_PATH)
sentiment_diario = sentiment.aggregate_daily_sentiment(raw_sent)

# Construir feature matrix completa
feature_matrix = features.build_feature_matrix(
    sp500_features_df,
    sentiment=sentiment_diario,
    include_sentiment=True,
    save_path=f"{PROCESSED_DIR}/feature_matrix.csv",
)

print(f"\nFeature matrix: {feature_matrix.shape}")
print(f"Cobertura de sentiment: {(feature_matrix['news_count'] > 0).mean()*100:.1f}% de los días")
feature_matrix[FEATURE_COLS_SENT].describe()

## Split temporal y escalado

Usamos el mismo split que en el notebook 03 (2008-2019 / 2020-2021 / 2022-2024) para que los resultados sean comparables entre experimentos. El scaler se fitea nuevamente sobre el train set con las 14 features.

In [ ]:
X_train, X_val, X_test, y_train, y_val, y_test = models.temporal_split(
    feature_matrix, features=FEATURE_COLS_SENT, target="target"
)

X_train_s, X_val_s, X_test_s, scaler_sent = models.scale_features(X_train, X_val, X_test)

## Experimento 3: XGBoost con Sentiment

Añadimos las 3 features de sentiment al XGBoost del Experimento 2. La comparación directa entre Exp 2 y Exp 3 mide el **aporte marginal del sentiment**.

Esperamos una mejora modesta pero consistente: el sentiment captura información cualitativa que los indicadores técnicos y macro no capturan. Sin embargo, si la mejora es mínima (< 0.5% en AUC), podría no justificar el costo computacional de FinBERT en producción.

In [ ]:
xgb_sent = models.train_xgboost(X_train_s, y_train, X_val_s, y_val)

metricas_exp3 = {}
for nombre, X, y in [("train", X_train_s, y_train), ("val", X_val_s, y_val), ("test", X_test_s, y_test)]:
    metricas_exp3[nombre] = models.evaluate_model(xgb_sent, X, y, nombre)

models.save_model(xgb_sent, scaler_sent, "xgboost_sentiment", MODELS_DIR)

## Experimento 4: MLP con Sentiment

El MLP (Multi-Layer Perceptron) es una red neuronal con 2 capas ocultas (128 y 64 neuronas). Puede capturar interacciones de mayor orden que XGBoost, pero requiere más datos y es más susceptible al overfitting.

**Arquitectura**: Input (14) → Hidden1 (128, ReLU) → Hidden2 (64, ReLU) → Output (2, Softmax)

**Early stopping**: detiene el entrenamiento cuando el error en validación no mejora en 15 épocas consecutivas, previniendo overfitting sin necesidad de fijar `max_iter` manualmente.

El MLP es nuestra única contribución de **deep learning** al ablation study — suficiente para cumplir con el requisito de la materia de incluir un modelo de aprendizaje profundo.

In [ ]:
mlp = models.train_mlp(X_train_s, y_train, X_val_s, y_val)

metricas_exp4 = {}
for nombre, X, y in [("train", X_train_s, y_train), ("val", X_val_s, y_val), ("test", X_test_s, y_test)]:
    metricas_exp4[nombre] = models.evaluate_model(mlp, X, y, nombre)

models.save_model(mlp, scaler_sent, "mlp_sentiment", MODELS_DIR)

## Tabla de ablation — Comparación de los 4 experimentos

Esta es la tabla central del proyecto. Permite responder directamente las preguntas de investigación:

1. **¿Técnicas + Macro vs solo Técnicas?** → comparar si agregar macro ayuda
2. **¿Qué aporta el sentiment?** → Exp 2 → Exp 3
3. **¿LR vs XGBoost?** → Exp 1 → Exp 2 (justifica la complejidad no-lineal)
4. **¿XGBoost vs MLP?** → Exp 3 → Exp 4 (justifica el deep learning)

En mercados financieros, **cualquier mejora sostenida en AUC es significativa**. Una mejora de 1-2% en AUC en el test set es un resultado sólido.

In [ ]:
# Cargar métricas de los experimentos 1 y 2 (guardadas por notebook 03)
# NOTA: si corriste el notebook 03 en la misma sesión, podés importar las variables.
# Si no, volvé a correr los modelos baselines aquí.

# Construir tabla de ablation
experimentos = [
    ("Exp 1", "Logistic Regression", "Técnicas + Macro"),
    ("Exp 2", "XGBoost",             "Técnicas + Macro"),
    ("Exp 3", "XGBoost",             "Técnicas + Macro + Sentiment"),
    ("Exp 4", "MLP (128-64)",        "Técnicas + Macro + Sentiment"),
]

# Reemplazar con las métricas reales de cada experimento
print("[Ver métricas en celdas anteriores y en el notebook 03 para Exp 1 y 2]")

# Ejemplo de tabla (completar con valores reales al ejecutar)
tabla_ablation = pd.DataFrame(experimentos, columns=["Experimento", "Modelo", "Features"])
display(tabla_ablation)

In [ ]:
# Curvas ROC — los 4 modelos en el mismo gráfico
# NOTA: lr y xgb del notebook 03 se pueden cargar con models.load_model()
lr_loaded,  scaler_lr  = models.load_model("logistic_regression", MODELS_DIR)
xgb_loaded, scaler_xgb = models.load_model("xgboost_baseline",    MODELS_DIR)

# Re-escalar test set con el scaler del baseline (11 features)
sp500_base = pd.read_csv(f"{PROCESSED_DIR}/sp500_features.csv", index_col=0, parse_dates=True)
sp500_base = sp500_base.dropna(subset=FEATURE_COLS_BASE)
test_base  = sp500_base[sp500_base.index.year >= 2022]
X_test_base = scaler_xgb.transform(test_base[FEATURE_COLS_BASE].values)
y_test_base = test_base["target"].values

fig, ax = plt.subplots(figsize=(9, 7))

from sklearn.metrics import roc_curve, auc
modelos_roc = [
    ("Exp 1: LR",             lr_loaded,  X_test_base,  y_test_base),
    ("Exp 2: XGBoost",        xgb_loaded, X_test_base,  y_test_base),
    ("Exp 3: XGBoost+Sent",   xgb_sent,   X_test_s,     y_test),
    ("Exp 4: MLP+Sent",       mlp,        X_test_s,     y_test),
]

for label, modelo, X, y in modelos_roc:
    y_score = modelo.predict_proba(X)[:, 1]
    fpr, tpr, _ = roc_curve(y, y_score)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, label=f"{label} (AUC={roc_auc:.3f})")

ax.plot([0, 1], [0, 1], "k--", label="Aleatorio (AUC=0.500)")
ax.set_xlabel("Tasa de Falsos Positivos")
ax.set_ylabel("Tasa de Verdaderos Positivos")
ax.set_title("Curvas ROC — Ablation Study (Test Set 2022-2024)")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

## SHAP: interpretabilidad del XGBoost

SHAP (SHapley Additive exPlanations) cuantifica el aporte de cada feature a cada predicción individual. A diferencia de la importancia de features estándar de XGBoost (que mide cuántas veces se usa un feature), SHAP mide el **impacto real en el output del modelo**.

Usamos el **SHAP TreeExplainer** que calcula valores exactos (no aproximaciones) aprovechando la estructura de árbol del XGBoost. Es el método más adecuado para este modelo.

El **summary plot** (beeswarm) muestra:
- **Eje Y**: features ordenadas por importancia global (suma de |SHAP values|)
- **Eje X**: impacto de esa feature en el output (positivo = contribuye a predecir "sube")
- **Color**: valor de la feature (rojo=alto, azul=bajo)

Esto nos permite responder: ¿qué features son más determinantes para predecir la dirección del S&P 500?

In [ ]:
models.shap_analysis(
    xgb_sent,
    X_train=X_train_s,
    X_test=X_test_s,
    feature_names=FEATURE_COLS_SENT,
)

## Conclusiones del ablation study

**¿Qué responden los experimentos?**

1. **¿Mejora el sentiment a XGBoost?** → comparar AUC de Exp 2 vs Exp 3 en el test set. Si la mejora es > 0.5% → el sentiment aporta información genuina.

2. **¿Vale la pena el MLP?** → comparar Exp 3 vs Exp 4. El MLP podría superar a XGBoost en training pero generalmente XGBoost es más robusto en series temporales financieras con pocas features tabulares.

3. **¿Qué features importan más según SHAP?** → típicamente esperamos que `return_1d`, `VIX` y `RSI_14` sean las más importantes.

**¿Por qué el mercado es difícil de predecir?**

La Hipótesis de Mercados Eficientes (EMH) sostiene que los precios ya reflejan toda la información disponible. Nuestros features son información pública — cualquier patrón explotable tiende a desaparecer a medida que los participantes del mercado lo descubren y actúan en consecuencia. Un AUC de 0.53-0.56 en el test set es un resultado realista y publicable (ver Reddy et al. CS224N 2023).

**Trabajo futuro:**
- LSTM o Transformer sobre secuencias de prices para capturar dependencias temporales
- Más fuentes de sentiment (Twitter/X, earnings calls)
- Features de opciones (implied volatility surface)
- Predicción de retorno cuantitativo en lugar de dirección binaria